# Modèle 1 : probabilité de retour à l'hôpital en moins de 30 jours

**Objectif :** prédire, pour chaque patient qui sort, la probabilité de revenir en moins de 30 jours (`readmitted == "<30"`).

**Comment l'utiliser dans Google Colab (gratuit, rien à installer) :**
1. Ouvrez [colab.research.google.com](https://colab.research.google.com), puis *Fichier → Importer un notebook* et choisissez ce fichier.
2. Menu *Exécution → Tout exécuter* (ou cellule par cellule avec Maj+Entrée).
3. À l'étape 1, une fenêtre vous demande d'importer `diabetic_data.csv`.
4. Pour un premier test rapide (1 à 2 minutes), mettez `SAMPLE = 5000` à l'étape 1. Remettez `None` pour l'entraînement complet.

**Temps estimé sur Colab (CPU gratuit), données complètes :** comparaison des modèles 10 à 25 min, réglage 5 à 15 min, calibration et test final 2 à 5 min.

| Étape | Contenu |
|---|---|
| 1 | Installation, réglages, import du fichier |
| 2 | Chargement et nettoyage |
| 3 | Création des variables |
| 4 | Séparation entraînement / test **par patient** |
| 5 | Étude comparative de plusieurs modèles |
| 6 | Réglage rapide du meilleur modèle |
| 7 | Calibration des probabilités |
| 8 | Test final |
| 9 | Niveaux de risque |
| 10 | Importance des variables |
| 11 | Sauvegarde et utilisation sur un nouveau patient |

## Étape 1. Installation, réglages, import du fichier

In [ ]:
import os, sys, time, subprocess, warnings, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---- Réglages ----
SAMPLE = None          # None = toutes les données ; 5000 = test rapide (nombre de patients)
SEED = 42
INSTALL_EXTRAS = True  # installe catboost si absent
DATA_PATH = "diabetic_data.csv"

def ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
        return True
    except ImportError:
        if not INSTALL_EXTRAS:
            return False
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        try:
            __import__(import_name or pkg); return True
        except ImportError:
            return False

HAS_LGBM = ensure("lightgbm")
HAS_XGB  = ensure("xgboost")
HAS_CAT  = ensure("catboost")
print("LightGBM:", HAS_LGBM, "| XGBoost:", HAS_XGB, "| CatBoost:", HAS_CAT)

# Import du fichier dans Colab (ignoré ailleurs)
try:
    from google.colab import files
    if not os.path.exists(DATA_PATH):
        print("Importez diabetic_data.csv :")
        files.upload()
except ImportError:
    pass

## Étape 2. Chargement et nettoyage

- `?` est une valeur manquante.
- On retire les patients **décédés ou en soins palliatifs** (`discharge_disposition_id` 11, 13, 14, 19, 20, 21) : ils ne peuvent pas être réadmis.
- La cible `y` vaut 1 si le patient est revenu en moins de 30 jours, sinon 0.

In [ ]:
df = pd.read_csv(DATA_PATH, na_values="?", low_memory=False)
print("Lignes au départ :", len(df))

EXCLUDE_DISCH = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(EXCLUDE_DISCH)].copy()
df["y"] = (df["readmitted"] == "<30").astype(int)

if SAMPLE:  # test rapide : on garde un échantillon de PATIENTS (pas de lignes)
    keep = df["patient_nbr"].drop_duplicates().sample(SAMPLE, random_state=SEED)
    df = df[df["patient_nbr"].isin(keep)].copy()

df = df.reset_index(drop=True)
print("Lignes utilisées :", len(df), "| patients :", df["patient_nbr"].nunique())
print("Taux de retour < 30 jours :", round(df["y"].mean(), 3))

## Étape 3. Création des variables

Les variables sont les informations connues **à la sortie** du patient. On n'utilise ni `readmitted` (c'est la cible), ni `encounter_id` / `patient_nbr` (identifiants), ni `weight` (97 % manquant).

Variables créées :
- `age_num` : milieu de la tranche d'âge ;
- `diag_*_grp` : les codes diagnostics regroupés en grandes familles ;
- `n_meds_diabete`, `n_meds_modifies` : nombre de médicaments diabète prescrits / modifiés ;
- `visites_anterieures` : hospitalisations + urgences + consultations de l'année précédente.

In [ ]:
MED_COLS = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
            "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
            "troglitazone","tolazamide","examide","citoglipton","insulin","glyburide-metformin",
            "glipizide-metformin","glimepiride-pioglitazone","metformin-rosiglitazone","metformin-pioglitazone"]

def icd9_group(code):
    if pd.isna(code): return "inconnu"
    c = str(code)
    if c.startswith(("E", "V")): return "autre"
    n = float(c)
    if int(n) == 250: return "diabete"
    if 390 <= n <= 459 or int(n) == 785: return "circulatoire"
    if 460 <= n <= 519 or int(n) == 786: return "respiratoire"
    if 520 <= n <= 579 or int(n) == 787: return "digestif"
    if 800 <= n <= 999: return "blessure"
    if 710 <= n <= 739: return "musculo-squelettique"
    if 580 <= n <= 629 or int(n) == 788: return "genito-urinaire"
    if 140 <= n <= 239: return "cancer"
    return "autre"

CAT = ["race","gender","payer_code","specialite","diag_1_grp","diag_2_grp","diag_3_grp","A1Cresult",
       "max_glu_serum","insulin","metformin","change","diabetesMed","admission_type_id",
       "admission_source_id","discharge_disposition_id"]
NUM = ["age_num","number_outpatient","number_emergency","number_inpatient","visites_anterieures",
       "number_diagnoses","time_in_hospital","num_lab_procedures","num_procedures","num_medications",
       "n_meds_diabete","n_meds_modifies"]

TOP_SPEC = df["medical_specialty"].value_counts().head(15).index.tolist()  # figé pour réutilisation

def build_features(d):
    """Transforme les colonnes brutes en variables du modèle (réutilisable pour un nouveau patient)."""
    d = d.copy()
    for c in ["diag_1","diag_2","diag_3"]:
        d[c + "_grp"] = d[c].map(icd9_group)
    d["age_num"] = d["age"].str.extract(r"\[(\d+)-")[0].astype(float) + 5
    d["n_meds_diabete"]  = (d[MED_COLS] != "No").sum(axis=1)
    d["n_meds_modifies"] = d[MED_COLS].isin(["Up", "Down"]).sum(axis=1)
    d["visites_anterieures"] = d["number_outpatient"] + d["number_emergency"] + d["number_inpatient"]
    d["specialite"] = d["medical_specialty"].where(d["medical_specialty"].isin(TOP_SPEC), "autre").fillna("inconnue")
    for c, v in [("payer_code","inconnu"),("race","inconnu"),("A1Cresult","non_mesure"),("max_glu_serum","non_mesure")]:
        d[c] = d[c].fillna(v)
    X = d[CAT + NUM].copy()
    for c in ["admission_type_id","admission_source_id","discharge_disposition_id"]:
        X[c] = X[c].astype(str)
    return X

X_raw = build_features(df)
CATEGORIES = {c: sorted(X_raw[c].astype(str).unique()) for c in CAT}

def apply_categories(X):
    X = X.copy()
    for c in CAT:
        X[c] = pd.Categorical(X[c].astype(str), categories=CATEGORIES[c])
    return X

X = apply_categories(X_raw)
y = df["y"].values
groups = df["patient_nbr"].values
print(X.shape, "| valeurs manquantes :", int(X.isna().sum().sum()))
X.head(3)

## Étape 4. Séparation entraînement / test **par patient**

Un patient a souvent plusieurs séjours : il ne doit jamais être à la fois dans l'entraînement et dans le test, sinon les résultats sont faussement bons.

- **20 % des patients** sont mis de côté comme **test final** : on n'y touche qu'à l'étape 8.
- Les 80 % restants (`dev`) servent à comparer et régler les modèles.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

dev_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
X_dev,  y_dev,  g_dev  = X.iloc[dev_idx].reset_index(drop=True),  y[dev_idx],  groups[dev_idx]
X_test, y_test, g_test = X.iloc[test_idx].reset_index(drop=True), y[test_idx], groups[test_idx]

print("dev :", len(X_dev), "lignes |", len(set(g_dev)), "patients | taux de retour", round(y_dev.mean(), 3))
print("test:", len(X_test), "lignes |", len(set(g_test)), "patients | taux de retour", round(y_test.mean(), 3))
assert len(set(g_dev) & set(g_test)) == 0, "Fuite : un patient est dans dev ET test"

## Étape 5. Étude comparative

Tous les modèles reçoivent exactement les mêmes données et les mêmes découpages (validation croisée à 5 plis, groupée par patient). On note pour chacun :

- **AUC** : capacité à classer les patients (0,5 = hasard) ;
- **PR-AUC** : utile quand la cible est rare (11 % de retours) ;
- **Lift top 10 %** : combien de fois plus de retours parmi les 10 % de patients jugés les plus à risque ;
- **Brier** : qualité des probabilités (plus petit = mieux) ;
- **Temps** d'entraînement.

La précision simple (accuracy) n'est pas utilisée : un modèle qui répond toujours « non » aurait environ 89 % de précision et serait inutile.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

def lift_top(y_true, p, frac=0.10):
    k = max(1, int(len(p) * frac))
    idx = np.argsort(-p)[:k]
    return y_true[idx].mean() / y_true.mean()

def prep(name, Xd):
    """CatBoost veut des textes pour les variables catégorielles ; les autres gardent le type 'category'."""
    if name.startswith("CatBoost"):
        Xd = Xd.copy()
        for c in CAT: Xd[c] = Xd[c].astype(str)
    return Xd

factories = {}

factories["LogReg"] = lambda: Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), CAT),
        ("num", StandardScaler(), NUM)])),
    ("clf", LogisticRegression(C=0.1, max_iter=2000))])

factories["HistGB"] = lambda: HistGradientBoostingClassifier(
    max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=5.0, min_samples_leaf=80,
    categorical_features="from_dtype", random_state=SEED)

if HAS_LGBM:
    from lightgbm import LGBMClassifier
    factories["LightGBM"] = lambda: LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=100, reg_lambda=5.0,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)

if HAS_XGB:
    from xgboost import XGBClassifier
    factories["XGBoost"] = lambda: XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4, min_child_weight=5, reg_lambda=5.0,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist", enable_categorical=True,
        random_state=SEED, n_jobs=-1, eval_metric="logloss")

if HAS_CAT:
    from catboost import CatBoostClassifier
    factories["CatBoost"] = lambda: CatBoostClassifier(
        iterations=400, learning_rate=0.06, depth=5, l2_leaf_reg=5.0, cat_features=CAT,
        random_seed=SEED, verbose=0, thread_count=-1)

results, oofs = {}, {}

def run_cv(name):
    oof, aucs, t0 = np.zeros(len(y_dev)), [], time.time()
    for tr, va in cv.split(X_dev, y_dev, groups=g_dev):
        m = factories[name]()
        m.fit(prep(name, X_dev.iloc[tr]), y_dev[tr])
        oof[va] = m.predict_proba(prep(name, X_dev.iloc[va]))[:, 1]
        aucs.append(roc_auc_score(y_dev[va], oof[va]))
    oofs[name] = oof
    results[name] = dict(AUC_moy=np.mean(aucs), AUC_ecart=np.std(aucs),
                         PR_AUC=average_precision_score(y_dev, oof),
                         Lift_top10=lift_top(y_dev, oof), Brier=brier_score_loss(y_dev, oof),
                         Temps_s=time.time() - t0)
    print(f"{name:14s} AUC {results[name]['AUC_moy']:.3f} ± {results[name]['AUC_ecart']:.3f} | "
          f"{results[name]['Temps_s']:.0f}s")

for name in list(factories):
    run_cv(name)

comparison = pd.DataFrame(results).T.sort_values("AUC_moy", ascending=False).round(3)
comparison

**Comment choisir :** prenez le meilleur AUC. Si deux modèles sont à moins d'un écart-type l'un de l'autre, prenez le **plus rapide et le plus simple**. Sur ce jeu de données, un AUC entre 0,63 et 0,70 est normal : les modèles sont souvent très proches.

## Étape 6. Réglage rapide du meilleur modèle

On teste 4 réglages de LightGBM (arbres peu profonds, forte régularisation : le signal est faible et le surapprentissage facile). Si LightGBM n'est pas disponible, on saute cette étape.

In [ ]:
if HAS_LGBM:
    grid = {
        "LGBM_a": dict(num_leaves=7,  min_child_samples=150, learning_rate=0.05, n_estimators=300, reg_lambda=5.0),
        "LGBM_b": dict(num_leaves=15, min_child_samples=100, learning_rate=0.03, n_estimators=400, reg_lambda=10.0),
        "LGBM_c": dict(num_leaves=31, min_child_samples=100, learning_rate=0.03, n_estimators=400, reg_lambda=10.0),
        "LGBM_d": dict(num_leaves=7,  min_child_samples=200, learning_rate=0.03, n_estimators=500, reg_lambda=10.0),
    }
    for gname, params in grid.items():
        factories[gname] = (lambda p=params: LGBMClassifier(
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=SEED, n_jobs=-1, verbose=-1, **p))
        run_cv(gname)

comparison = pd.DataFrame(results).T.sort_values("AUC_moy", ascending=False).round(3)
display(comparison)

BEST_NAME = comparison.index[0]     # ou forcez-le, par exemple : BEST_NAME = "CatBoost"
print("Modèle retenu :", BEST_NAME)

## Étape 7. Calibration des probabilités

Le planning des lits **additionne** des probabilités. Si le modèle dit 20 % alors que la réalité est 11 %, le nombre de lits réservé sera faux. On corrige avec une régression isotonique, ajustée sur les prédictions **hors entraînement** (out-of-fold) de l'étape 5.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

oof_raw = oofs[BEST_NAME]
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(oof_raw, y_dev)

fx, fy = calibration_curve(y_dev, oof_raw, n_bins=10, strategy="quantile")
plt.figure(figsize=(4.5, 4.5))
plt.plot([0, max(fx.max(), fy.max())], [0, max(fx.max(), fy.max())], "--", color="grey", label="parfait")
plt.plot(fx, fy, "o-", label=f"{BEST_NAME} (avant calibration)")
plt.xlabel("Probabilité prédite"); plt.ylabel("Taux réel de retour"); plt.legend(); plt.title("Calibration (validation croisée)")
plt.show()
print("Brier avant calibration :", round(brier_score_loss(y_dev, oof_raw), 4))

## Étape 8. Test final (une seule fois)

On entraîne le modèle retenu sur **tous** les patients de `dev`, puis on l'évalue une seule fois sur le test mis de côté à l'étape 4.

In [ ]:
final_model = factories[BEST_NAME]()
final_model.fit(prep(BEST_NAME, X_dev), y_dev)

p_raw = final_model.predict_proba(prep(BEST_NAME, X_test))[:, 1]
p_cal = iso.predict(p_raw)

def report(tag, p):
    print(f"{tag:12s} AUC {roc_auc_score(y_test, p):.3f} | PR-AUC {average_precision_score(y_test, p):.3f} | "
          f"Lift top10 {lift_top(y_test, p):.2f} | Brier {brier_score_loss(y_test, p):.4f}")

print("Taux réel de retour dans le test :", round(y_test.mean(), 3))
report("brut", p_raw)
report("calibré", p_cal)

# Taux réel observé dans les 10 % de patients les plus à risque
k = int(len(p_cal) * 0.10); top = np.argsort(-p_cal)[:k]
print(f"Parmi les 10 % les plus à risque : {y_test[top].mean():.1%} sont revenus (moyenne : {y_test.mean():.1%})")

# Probabilité moyenne prédite vs taux réel, par dixième de patients
tab = pd.DataFrame({"p": p_cal, "y": y_test})
tab["decile"] = pd.qcut(tab["p"].rank(method="first"), 10, labels=range(1, 11))
print(tab.groupby("decile").agg(prob_moyenne=("p", "mean"), taux_reel=("y", "mean"), n=("y", "size")).round(3))

**Comment lire le résultat :** l'AUC doit être clairement au-dessus de 0,5. Le lift doit dépasser 1 (le modèle repère mieux que le hasard). Dans le tableau par dixième, la probabilité moyenne et le taux réel doivent être proches : c'est la preuve que la calibration est bonne.

## Étape 9. Niveaux de risque

Trois niveaux (faible, moyen, élevé), fixés sur les prédictions hors entraînement : 70 % des patients sont « faible », 20 % « moyen », 10 % « élevé ». La clinique peut ajuster ces seuils selon le nombre de patients qu'elle peut suivre de près.

In [ ]:
p_oof_cal = iso.predict(oof_raw)
thr_med, thr_high = np.percentile(p_oof_cal, [70, 90])
print(f"Moyen à partir de {thr_med:.1%} | Élevé à partir de {thr_high:.1%}")

level = np.where(p_cal >= thr_high, "élevé", np.where(p_cal >= thr_med, "moyen", "faible"))
risk_tab = pd.DataFrame({"niveau": level, "y": y_test}).groupby("niveau").agg(
    patients=("y", "size"), taux_reel_de_retour=("y", "mean")).loc[["faible", "moyen", "élevé"]].round(3)
risk_tab

## Étape 10. Importance des variables

On mesure de combien l'AUC baisse quand on mélange une variable : plus la baisse est grande, plus la variable compte.

In [ ]:
from sklearn.inspection import permutation_importance

n_imp = min(5000, len(X_test))
idx = np.random.RandomState(SEED).choice(len(X_test), n_imp, replace=False)
pi = permutation_importance(final_model, prep(BEST_NAME, X_test.iloc[idx]), y_test[idx],
                            scoring="roc_auc", n_repeats=3, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
imp.head(12).iloc[::-1].plot.barh(figsize=(6, 4), title="Variables les plus utiles (baisse d'AUC)")
plt.show()
imp.head(12).round(4)

## Étape 11. Sauvegarde et utilisation sur un nouveau patient

In [ ]:
artifact = dict(model=final_model, calibrator=iso, best_name=BEST_NAME, cat=CAT, num=NUM,
                categories=CATEGORIES, top_spec=TOP_SPEC, thr_med=float(thr_med), thr_high=float(thr_high))
joblib.dump(artifact, "model_readmission_30d.joblib")
print("Modèle sauvegardé : model_readmission_30d.joblib")

try:
    from google.colab import files
    files.download("model_readmission_30d.joblib")
except ImportError:
    pass

def predict_patients(raw_df):
    """raw_df : tableau avec les MÊMES colonnes que diabetic_data.csv. Renvoie risque et niveau."""
    Xn = apply_categories(build_features(raw_df))
    p = iso.predict(final_model.predict_proba(prep(BEST_NAME, Xn))[:, 1])
    lvl = np.where(p >= thr_high, "élevé", np.where(p >= thr_med, "moyen", "faible"))
    return pd.DataFrame({"risque_retour_30j": p.round(3), "niveau": lvl}, index=raw_df.index)

# Exemple : 5 patients du jeu de test
predict_patients(df.iloc[test_idx[:5]])